# Módulo 3 · Clasificación supervisada

**Notebook de práctica** · Escuela de Machine Learning Honduras 2026

En este notebook vas a practicar lo que vimos en las diapositivas:

1. ¿Qué es clasificar?
2. Datos de entrenamiento y de prueba
3. k-vecinos más cercanos (kNN)
4. Árboles de decisión
5. Random Forest
6. Máquinas de vectores de soporte (SVM)
7. Naive Bayes
8. Cómo evaluar un clasificador
9. Comparar modelos
10. Reto final

**Cómo usarlo**

- Corre las celdas en orden, de arriba hacia abajo (`Shift + Enter`).
- Cuando veas **✏️ Ejercicio**, escribe tu código donde dice `# TU CÓDIGO AQUÍ`.
- Las celdas con `assert` revisan tu respuesta. Si no sale ningún error, ¡vas bien!
- Si algo falla, lee el mensaje de error con calma: casi siempre dice qué pasó.

In [ ]:
# Librerías que vamos a usar
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris, load_breast_cancer, make_moons, make_circles, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
                             precision_score, recall_score, f1_score,
                             classification_report, roc_curve, roc_auc_score)

plt.rcParams["figure.figsize"] = (6, 4)
SEMILLA = 42  # para que todos obtengamos los mismos resultados
print("¡Todo listo!")

In [ ]:
# Función de ayuda: dibuja la frontera de decisión de un modelo con 2 variables.
# No necesitas entenderla por dentro; solo úsala.
def dibujar_frontera(modelo, X, y, titulo="", ax=None):
    if ax is None:
        ax = plt.gca()
    x0, x1 = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y0, y1 = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x0, x1, 300), np.linspace(y0, y1, 300))
    Z = modelo.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=25)
    ax.set_title(titulo)

## 1. ¿Qué es clasificar?

Empecemos con el ejemplo de las diapositivas. Tenemos datos de alumnos de semestres pasados:
cuántas horas estudian por semana, su asistencia y si aprobaron o no.

> Los datos de este ejemplo son inventados, solo para practicar.

In [ ]:
alumnos = pd.DataFrame({
    "nombre":     ["Luis", "Pedro", "Diego", "Carla", "Sofía", "Marta"],
    "horas":      [2, 3, 5, 6, 7, 8],
    "asistencia": [60, 95, 70, 90, 75, 90],
    "aprobo":     [0, 0, 0, 1, 1, 1],   # 1 = sí, 0 = no
})
alumnos

In [ ]:
colores = alumnos["aprobo"].map({0: "tab:blue", 1: "tab:orange"})
plt.scatter(alumnos["horas"], alumnos["asistencia"], c=colores, s=120, edgecolor="k")
for _, fila in alumnos.iterrows():
    plt.annotate(fila["nombre"], (fila["horas"] + 0.15, fila["asistencia"] + 1))

# Ana: la alumna nueva
plt.scatter([6], [85], marker="*", s=300, c="black")
plt.annotate("Ana (?)", (6.15, 86))

plt.xlabel("Horas de estudio por semana")
plt.ylabel("Asistencia (%)")
plt.title("Azul = no aprobó · Naranja = aprobó")
plt.show()

Cada alumno es un **punto**. Sus datos (horas y asistencia) son sus **coordenadas**.

Clasificar es encontrar una **regla** que diga de qué lado de la frontera cae un punto nuevo.

Una regla sencilla que funciona con estos datos es:

$$\text{aprueba si } \text{horas} + \frac{\text{asistencia}}{20} \geq 9$$

### ✏️ Ejercicio 1

Escribe la función `regla_aprueba(horas, asistencia)` que regrese `1` si el alumno aprueba según la regla
y `0` si no. Luego úsala para Ana (6 h, 85 %) y para Jorge (5 h, 90 %).

In [ ]:
def regla_aprueba(horas, asistencia):
    # TU CÓDIGO AQUÍ
    pass

print("Ana:", regla_aprueba(6, 85))
print("Jorge:", regla_aprueba(5, 90))

In [ ]:
# Revisión: la regla debe acertar en los 6 alumnos de la tabla
predicciones = [regla_aprueba(h, a) for h, a in zip(alumnos["horas"], alumnos["asistencia"])]
assert predicciones == list(alumnos["aprobo"]), "La regla no coincide con la tabla. Revisa la fórmula."
print("✅ ¡Correcto! Tu regla clasifica bien a los 6 alumnos.")

**Pregunta para pensar:** Jorge queda muy cerca de la frontera. ¿Confiarías en la respuesta?
Con pocos datos, una regla escrita a mano funciona. Con miles de alumnos y muchas variables,
necesitamos que la computadora **aprenda la regla sola**. Eso es lo que hacen los clasificadores.

### Vocabulario

| En el ejemplo | Nombre técnico | En código |
|---|---|---|
| Cada alumno | observación | una fila |
| Horas y asistencia | variables (*features*) | `X` |
| ¿Aprobó? | etiqueta o clase | `y` |
| Tu regla | clasificador | `modelo` |

In [ ]:
X = alumnos[["horas", "asistencia"]].values   # matriz de variables
y = alumnos["aprobo"].values                  # vector de etiquetas
print("Forma de X:", X.shape, "→ 6 alumnos, 2 variables")
print("Forma de y:", y.shape)

## 2. Datos de entrenamiento y de prueba

Un buen modelo debe acertar con datos **que no ha visto**. Por eso separamos los datos en dos partes:

- **Entrenamiento (70 %)**: el modelo aprende de aquí.
- **Prueba (30 %)**: la guardamos y solo la usamos al final para medir qué tan bien funciona.

Vamos a usar el conjunto **Iris**: 150 flores de 3 especies, con 4 medidas cada una.

In [ ]:
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
print("Variables:", iris.feature_names)
print("Clases:", [str(c) for c in iris.target_names])
print("Forma:", X_iris.shape)

In [ ]:
# ✏️ Ejercicio 2: separa X_iris y y_iris en entrenamiento (70 %) y prueba (30 %).
# Usa train_test_split con test_size=0.3, random_state=SEMILLA y stratify=y_iris.
X_ent, X_pru, y_ent, y_pru = None, None, None, None  # TU CÓDIGO AQUÍ

print("Entrenamiento:", X_ent.shape, "· Prueba:", X_pru.shape)

In [ ]:
assert X_ent.shape == (105, 4) and X_pru.shape == (45, 4), "Revisa test_size=0.3"
print("✅ ¡Bien! 105 flores para entrenar y 45 para probar.")

`stratify=y_iris` hace que las 3 especies aparezcan en la misma proporción en ambas partes.
Es muy útil cuando una clase tiene pocos ejemplos.

## 3. k-vecinos más cercanos (kNN)

**Idea:** «Dime con quién andas y te diré quién eres».

Para clasificar un punto nuevo:

1. Calcula la distancia a **todos** los puntos de entrenamiento.
2. Toma los **k** más cercanos.
3. Predice la clase que más se repite entre ellos.

### 3.1 kNN desde cero

Esta es la misma implementación de las diapositivas. Léela con calma.

In [ ]:
class KNN:
    def __init__(self, k=5):
        self.k = k

    def fit(self, X, y):
        # kNN no aprende nada: solo guarda los datos
        self.X = np.asarray(X)
        self.y = np.asarray(y)
        return self

    def predict(self, X_nuevo):
        X_nuevo = np.asarray(X_nuevo)
        # Distancia de cada punto nuevo a cada punto de entrenamiento → matriz (m, n)
        D = np.linalg.norm(X_nuevo[:, None, :] - self.X[None, :, :], axis=2)
        # Índices de los k vecinos más cercanos
        vecinos = np.argsort(D, axis=1)[:, :self.k]
        # Voto de la mayoría
        return np.array([np.bincount(fila).argmax() for fila in self.y[vecinos]])

In [ ]:
mi_knn = KNN(k=5).fit(X_ent, y_ent)
pred_mio = mi_knn.predict(X_pru)

knn_sk = KNeighborsClassifier(n_neighbors=5).fit(X_ent, y_ent)
pred_sk = knn_sk.predict(X_pru)

print(f"Nuestro kNN:      {accuracy_score(y_pru, pred_mio):.3f}")
print(f"kNN scikit-learn: {accuracy_score(y_pru, pred_sk):.3f}")

¡Nuestro kNN y el de scikit-learn deberían dar la misma exactitud!

### 3.2 ¿Qué k elegir?

- Con **k = 1** el modelo memoriza: acierta 100 % en entrenamiento, pero puede fallar con datos nuevos.
- Con **k muy grande** el modelo se vuelve demasiado simple.

Veámoslo con datos en forma de lunas.

In [ ]:
X_lunas, y_lunas = make_moons(n_samples=300, noise=0.3, random_state=SEMILLA)
Xl_ent, Xl_pru, yl_ent, yl_pru = train_test_split(X_lunas, y_lunas, test_size=0.3, random_state=SEMILLA)

fig, ejes = plt.subplots(1, 3, figsize=(15, 4))
for ax, k in zip(ejes, [1, 15, 100]):
    m = KNeighborsClassifier(n_neighbors=k).fit(Xl_ent, yl_ent)
    titulo = f"k = {k} · entren. {m.score(Xl_ent, yl_ent):.2f} · prueba {m.score(Xl_pru, yl_pru):.2f}"
    dibujar_frontera(m, Xl_ent, yl_ent, titulo, ax)
plt.show()

### ✏️ Ejercicio 3

Calcula la exactitud de **entrenamiento** y de **prueba** para k = 1, 2, …, 50 con los datos de lunas.
Guarda los resultados en las listas `exact_ent` y `exact_pru`. Luego corre la gráfica.

In [ ]:
valores_k = range(1, 51)
exact_ent, exact_pru = [], []
for k in valores_k:
    # TU CÓDIGO AQUÍ: entrena un KNeighborsClassifier con n_neighbors=k
    # y agrega a las listas la exactitud en entrenamiento y en prueba (usa .score)
    pass

In [ ]:
assert len(exact_pru) == 50, "Debes tener 50 valores en cada lista"
plt.plot(valores_k, exact_ent, label="entrenamiento")
plt.plot(valores_k, exact_pru, label="prueba")
plt.xlabel("k"); plt.ylabel("exactitud"); plt.legend(); plt.title("kNN: exactitud según k")
plt.show()
print("Mejor k en prueba:", valores_k[int(np.argmax(exact_pru))])

⚠️ **Cuidado:** acabamos de elegir k mirando el conjunto de **prueba**. Eso es hacer trampa:
la prueba debe usarse solo al final. La forma correcta es la **validación cruzada**.

### 3.3 Validación cruzada

Divide el entrenamiento en 5 partes. Entrena con 4 y evalúa con la que sobra. Repite 5 veces y promedia.

In [ ]:
resultados = []
for k in range(1, 51, 2):
    puntajes = cross_val_score(KNeighborsClassifier(n_neighbors=k), Xl_ent, yl_ent, cv=5)
    resultados.append((k, puntajes.mean()))

mejor_k, mejor_cv = max(resultados, key=lambda r: r[1])
print(f"Mejor k según validación cruzada: {mejor_k} (exactitud CV = {mejor_cv:.3f})")

final = KNeighborsClassifier(n_neighbors=mejor_k).fit(Xl_ent, yl_ent)
print(f"Exactitud en prueba (se mide UNA sola vez, al final): {final.score(Xl_pru, yl_pru):.3f}")

### 3.4 ¡Escala tus datos!

kNN usa distancias. Si una variable va de 0 a 1000 y otra de 0 a 1, la primera domina todo.
Veámoslo con el conjunto de **cáncer de mama** (569 tumores, 30 medidas).

In [ ]:
cancer = load_breast_cancer()
X_c, y_c = cancer.data, cancer.target   # 0 = maligno, 1 = benigno
Xc_ent, Xc_pru, yc_ent, yc_pru = train_test_split(X_c, y_c, test_size=0.3, random_state=SEMILLA, stratify=y_c)

print("Rango de 'mean area':      ", X_c[:, 3].min().round(1), "a", X_c[:, 3].max().round(1))
print("Rango de 'mean smoothness':", X_c[:, 4].min().round(3), "a", X_c[:, 4].max().round(3))

In [ ]:
# ✏️ Ejercicio 4: entrena dos kNN con k = 7 sobre Xc_ent, yc_ent
#   1) sin escalar
#   2) con make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7))
# y compara su exactitud en la prueba.
sin_escalar = None  # TU CÓDIGO AQUÍ
con_escalar = None  # TU CÓDIGO AQUÍ

print(f"Sin escalar: {sin_escalar.score(Xc_pru, yc_pru):.3f}")
print(f"Con escalar: {con_escalar.score(Xc_pru, yc_pru):.3f}")

`StandardScaler` resta la media y divide entre la desviación estándar de cada variable.
Así todas pesan parecido. El `pipeline` aprende la escala **solo** con el entrenamiento, sin «espiar» la prueba.

## 4. Árboles de decisión

Un árbol hace **preguntas de sí o no**, como «¿largo del pétalo ≤ 2.45?».
En cada nodo elige la pregunta que deja los grupos más **puros**.

La **impureza de Gini** mide qué tan mezclado está un grupo:

$$G = 1 - \sum_k p_k^2$$

donde $p_k$ es la proporción de la clase $k$. Un grupo puro tiene $G = 0$.

### ✏️ Ejercicio 5

Escribe la función `gini(conteos)` que reciba una lista con cuántos ejemplos hay de cada clase.
Por ejemplo, un nodo con 40 setosas y 10 versicolor es `gini([40, 10])`.

In [ ]:
def gini(conteos):
    # TU CÓDIGO AQUÍ
    # Pista: primero calcula las proporciones p = conteos / total
    pass

print(round(gini([40, 10]), 4))

In [ ]:
assert abs(gini([40, 10]) - 0.32) < 1e-9, "Para [40, 10] el resultado debe ser 0.32"
assert gini([50, 0]) == 0, "Un nodo puro tiene Gini = 0"
assert abs(gini([50, 50, 50]) - 2/3) < 1e-9, "Iris completo (3 clases balanceadas) tiene Gini = 2/3"
print("✅ ¡Perfecto!")

Otra medida es la **entropía**: $H = -\sum_k p_k \log_2 p_k$. En la práctica, Gini y entropía eligen cortes muy parecidos.

### 4.1 Un árbol pequeño con Iris

In [ ]:
arbol = DecisionTreeClassifier(max_depth=3, random_state=SEMILLA).fit(X_ent, y_ent)
print(f"Exactitud en prueba: {arbol.score(X_pru, y_pru):.3f}")

plt.figure(figsize=(14, 7))
plot_tree(arbol, feature_names=iris.feature_names, class_names=list(iris.target_names),
          filled=True, rounded=True, fontsize=9)
plt.show()

Cada caja muestra la pregunta, el Gini, cuántas muestras llegan y cuántas hay de cada clase.
¡Un árbol se puede leer como una lista de reglas!

### 4.2 Profundidad y sobreajuste

Sin límite de profundidad, el árbol sigue preguntando hasta **memorizar** los datos.

In [ ]:
# ✏️ Ejercicio 6: igual que en el ejercicio 3, pero ahora con DecisionTreeClassifier.
# Prueba max_depth de 1 a 15 con los datos de lunas (Xl_ent, yl_ent, Xl_pru, yl_pru)
# y grafica la exactitud de entrenamiento y de prueba.
profundidades = range(1, 16)
ent, pru = [], []
# TU CÓDIGO AQUÍ

**Pregunta:** ¿a partir de qué profundidad la exactitud de entrenamiento llega a 1.0? ¿Qué pasa con la de prueba?
Esa diferencia es el **sobreajuste**.

## 5. Random Forest

Un árbol solo es **inestable**: si cambian un poco los datos, cambia mucho el árbol.
Random Forest entrena **muchos árboles distintos** y los pone a votar:

1. **Bagging:** cada árbol aprende con una muestra al azar de las filas (con reemplazo).
2. **Subespacios:** en cada corte, solo se consideran algunas variables al azar.
3. **Votación:** gana la clase con más votos.

In [ ]:
bosque = RandomForestClassifier(n_estimators=300, oob_score=True, random_state=SEMILLA)
bosque.fit(Xc_ent, yc_ent)

print(f"Exactitud OOB (sin usar la prueba): {bosque.oob_score_:.3f}")
print(f"Exactitud en prueba:                {bosque.score(Xc_pru, yc_pru):.3f}")

**¿Qué es OOB?** Cada árbol no ve más o menos un 37 % de las filas. Con esas filas «sobrantes»
podemos medir el error **gratis**, sin apartar más datos.

### 5.1 ¿Qué variables importan más?

In [ ]:
importancias = pd.Series(bosque.feature_importances_, index=cancer.feature_names).sort_values()
importancias.tail(10).plot.barh(figsize=(7, 4), title="Las 10 variables más importantes")
plt.show()

### 5.2 Un árbol vs. un bosque: estabilidad

Vamos a repetir el experimento con 30 particiones distintas de los datos.

In [ ]:
res_arbol, res_bosque = [], []
for semilla in range(30):
    a, b, c, d = train_test_split(X_c, y_c, test_size=0.3, random_state=semilla, stratify=y_c)
    res_arbol.append(DecisionTreeClassifier(random_state=0).fit(a, c).score(b, d))
    res_bosque.append(RandomForestClassifier(n_estimators=100, random_state=0).fit(a, c).score(b, d))

print(f"Árbol solo:    {np.mean(res_arbol):.3f} ± {np.std(res_arbol):.3f}")
print(f"Random Forest: {np.mean(res_bosque):.3f} ± {np.std(res_bosque):.3f}")

plt.boxplot([res_arbol, res_bosque])
plt.xticks([1, 2], ["Árbol", "Random Forest"])
plt.ylabel("exactitud en prueba"); plt.title("30 particiones distintas")
plt.show()

El bosque no solo acierta más en promedio: sus resultados varían **mucho menos**. Eso es reducir la varianza.

## 6. Máquinas de vectores de soporte (SVM)

La SVM busca la frontera con el **mayor margen** entre las clases.
Solo los puntos más cercanos a la frontera (los **vectores de soporte**) la definen.

Dos parámetros importantes:

- **C**: C grande castiga los errores (margen angosto). C chico tolera errores (margen ancho).
- **kernel**: `"linear"` para fronteras rectas, `"rbf"` para fronteras curvas.

In [ ]:
X_circ, y_circ = make_circles(n_samples=300, noise=0.1, factor=0.4, random_state=SEMILLA)

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
for ax, kernel in zip(ejes, ["linear", "rbf"]):
    m = SVC(kernel=kernel, C=1).fit(X_circ, y_circ)
    dibujar_frontera(m, X_circ, y_circ, f"kernel = {kernel} · exactitud {m.score(X_circ, y_circ):.2f}", ax)
plt.show()

Con datos en círculos, **ninguna recta** sirve. El kernel RBF sí puede separar las clases.

### ✏️ Ejercicio 7

Entrena una SVM lineal con los datos de lunas (`Xl_ent`, `yl_ent`) para estos valores de C:
`0.01, 0.1, 1, 10, 100`. Para cada uno imprime **cuántos vectores de soporte** tiene
(usa `len(modelo.support_)`) y la exactitud en prueba.

In [ ]:
for C in [0.01, 0.1, 1, 10, 100]:
    # TU CÓDIGO AQUÍ
    pass

**Pregunta:** ¿por qué con C chico hay **más** vectores de soporte? (Pista: piensa en qué tan ancho es el margen.)

> En la práctica: con SVM **siempre escala** los datos, igual que con kNN.

## 7. Naive Bayes

Usa el **teorema de Bayes** y supone que las variables son **independientes** dada la clase:

$$P(\text{clase} \mid x) \;\propto\; P(\text{clase}) \cdot \prod_j P(x_j \mid \text{clase})$$

### 7.1 El ejemplo del spam, a mano

Tenemos 100 correos: 40 son spam y 60 son normales (números inventados).

| palabra | aparece en spam | aparece en normales |
|---|---|---|
| gratis | 20 de 40 | 3 de 60 |
| reunión | 4 de 40 | 18 de 60 |

Llega un correo con las palabras **«gratis»** y **«reunión»**. ¿Es spam?

### ✏️ Ejercicio 8

Calcula el puntaje de cada clase y la probabilidad de que sea spam.

In [ ]:
p_spam, p_normal = 40 / 100, 60 / 100
puntaje_spam = None    # TU CÓDIGO AQUÍ: P(spam) · P(gratis | spam) · P(reunión | spam)
puntaje_normal = None  # TU CÓDIGO AQUÍ
prob_spam = None       # TU CÓDIGO AQUÍ: normaliza para que las dos probabilidades sumen 1

print(f"Puntaje spam:   {puntaje_spam:.4f}")
print(f"Puntaje normal: {puntaje_normal:.4f}")
print(f"P(spam | correo) = {prob_spam:.2f}")

In [ ]:
assert abs(puntaje_spam - 0.02) < 1e-9 and abs(puntaje_normal - 0.009) < 1e-9
assert abs(prob_spam - 0.6897) < 1e-3
print("✅ ¡Correcto! El correo es spam con probabilidad ≈ 69 %.")

### 7.2 El problema del cero y el suavizado de Laplace

Imagina que la palabra «criptomoneda» aparece en 6 correos spam pero en **0** correos normales.
Entonces $P(\text{criptomoneda} \mid \text{normal}) = 0$ y **todo el producto se vuelve cero**,
sin importar las demás palabras.

La solución es el **suavizado de Laplace**: sumar 1 a cada conteo.

In [ ]:
def prob_palabra(conteo, total, alfa=1):
    # Suavizado de Laplace para una palabra que aparece o no aparece (2 resultados posibles)
    return (conteo + alfa) / (total + 2 * alfa)

print("Sin suavizado:", prob_palabra(0, 60, alfa=0))
print("Con suavizado:", round(prob_palabra(0, 60, alfa=1), 4))

### 7.3 Naive Bayes gaussiano

Para variables continuas, se supone que cada variable sigue una **distribución normal** dentro de cada clase.

In [ ]:
nb = GaussianNB().fit(X_ent, y_ent)
print(f"Naive Bayes en Iris · exactitud en prueba: {nb.score(X_pru, y_pru):.3f}")

nb_lunas = GaussianNB().fit(Xl_ent, yl_ent)
dibujar_frontera(nb_lunas, Xl_ent, yl_ent, f"Naive Bayes en lunas · prueba {nb_lunas.score(Xl_pru, yl_pru):.2f}")
plt.show()

Naive Bayes es **muy rápido** y funciona bien con texto. Su frontera es sencilla, por eso
le cuesta con formas como las lunas.

## 8. ¿Cómo evaluar un clasificador?

### 8.1 Matriz de confusión

|  | predice positivo | predice negativo |
|---|---|---|
| **real positivo** | VP (verdadero positivo) | FN (falso negativo) |
| **real negativo** | FP (falso positivo) | VN (verdadero negativo) |

En el conjunto de cáncer, vamos a tratar **maligno** como la clase positiva (la que queremos detectar).

In [ ]:
# En los datos originales 0 = maligno. Lo cambiamos para que 1 = maligno.
yc_ent_m = 1 - yc_ent
yc_pru_m = 1 - yc_pru

modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xc_ent, yc_ent_m)
pred = modelo.predict(Xc_pru)

ConfusionMatrixDisplay(confusion_matrix(yc_pru_m, pred), display_labels=["benigno", "maligno"]).plot(cmap="Blues")
plt.show()

### ✏️ Ejercicio 9

Con la matriz de confusión, calcula **a mano** (con VP, FP, FN, VN) la precisión, el recall y el F1.
Después compáralos con las funciones de scikit-learn.

- Precisión = VP / (VP + FP): de los que marqué malignos, ¿cuántos lo eran?
- Recall = VP / (VP + FN): de los malignos reales, ¿cuántos encontré?
- F1 = 2 · P · R / (P + R)

In [ ]:
VN, FP, FN, VP = confusion_matrix(yc_pru_m, pred).ravel()
precision = None  # TU CÓDIGO AQUÍ
recall = None     # TU CÓDIGO AQUÍ
f1 = None         # TU CÓDIGO AQUÍ

print(f"A mano       · precisión {precision:.3f} · recall {recall:.3f} · F1 {f1:.3f}")
print(f"scikit-learn · precisión {precision_score(yc_pru_m, pred):.3f} · "
      f"recall {recall_score(yc_pru_m, pred):.3f} · F1 {f1_score(yc_pru_m, pred):.3f}")

In [ ]:
assert abs(precision - precision_score(yc_pru_m, pred)) < 1e-9
assert abs(recall - recall_score(yc_pru_m, pred)) < 1e-9
assert abs(f1 - f1_score(yc_pru_m, pred)) < 1e-9
print("✅ ¡Tus cálculos coinciden con scikit-learn!")

En medicina, un **falso negativo** (decir «benigno» a un tumor maligno) cuesta mucho más que un falso positivo.
Por eso aquí nos importa mucho el **recall**.

### 8.2 Mover el umbral

El modelo da una **probabilidad**. Por defecto, si es ≥ 0.5 predice «maligno».
Si bajamos el umbral, encontramos más malignos (sube el recall), pero nos equivocamos más veces (baja la precisión).

In [ ]:
prob = modelo.predict_proba(Xc_pru)[:, 1]
for umbral in [0.1, 0.3, 0.5, 0.7, 0.9]:
    p = (prob >= umbral).astype(int)
    print(f"umbral {umbral:.1f} · precisión {precision_score(yc_pru_m, p):.3f} · "
          f"recall {recall_score(yc_pru_m, p):.3f}")

### 8.3 Curva ROC y AUC

La curva ROC muestra **todos los umbrales a la vez**: tasa de verdaderos positivos contra tasa de falsos positivos.
El **AUC** es el área bajo la curva: 0.5 = azar, 1 = perfecto.

In [ ]:
fpr, tpr, _ = roc_curve(yc_pru_m, prob)
plt.plot(fpr, tpr, label=f"Regresión logística (AUC = {roc_auc_score(yc_pru_m, prob):.3f})")
plt.plot([0, 1], [0, 1], "--", color="gray", label="azar")
plt.xlabel("Tasa de falsos positivos"); plt.ylabel("Tasa de verdaderos positivos")
plt.legend(); plt.title("Curva ROC")
plt.show()

### 8.4 La trampa de la exactitud con clases desbalanceadas

Imagina un problema de fraude donde solo el **2 %** de los casos son fraude.
Un modelo que **siempre** dice «no es fraude» tendría 98 % de exactitud... ¡y no detecta ningún fraude!

In [ ]:
X_f, y_f = make_classification(n_samples=5000, n_features=10, weights=[0.98], flip_y=0.01, random_state=SEMILLA)
Xf_ent, Xf_pru, yf_ent, yf_pru = train_test_split(X_f, y_f, test_size=0.3, random_state=SEMILLA, stratify=y_f)
print(f"Proporción de fraudes: {y_f.mean():.3f}")

tonto = DummyClassifier(strategy="most_frequent").fit(Xf_ent, yf_ent)
bosque_f = RandomForestClassifier(n_estimators=200, random_state=SEMILLA).fit(Xf_ent, yf_ent)

for nombre, m in [("Siempre 'no fraude'", tonto), ("Random Forest", bosque_f)]:
    p = m.predict(Xf_pru)
    print(f"{nombre:<20} · exactitud {accuracy_score(yf_pru, p):.3f} · "
          f"recall {recall_score(yf_pru, p, zero_division=0):.3f} · F1 {f1_score(yf_pru, p, zero_division=0):.3f}")

**Lección:** con clases desbalanceadas, la exactitud **engaña**. Revisa siempre la matriz de confusión,
el recall, la precisión y el F1.

In [ ]:
print(classification_report(yf_pru, bosque_f.predict(Xf_pru), target_names=["normal", "fraude"]))

## 9. Comparar modelos

Vamos a comparar todos los modelos con **validación cruzada** en el conjunto de cáncer.
Los que usan distancias (kNN, SVM) y la regresión logística van con escalado.

### ✏️ Ejercicio 10

Completa el diccionario `modelos` con los 6 clasificadores que vimos y corre la comparación.

In [ ]:
modelos = {
    "Regresión logística": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "kNN (k = 7)":         None,  # TU CÓDIGO AQUÍ (con escalado)
    "Árbol":               None,  # TU CÓDIGO AQUÍ
    "Random Forest":       None,  # TU CÓDIGO AQUÍ (300 árboles)
    "SVM (RBF)":           None,  # TU CÓDIGO AQUÍ (con escalado)
    "Naive Bayes":         None,  # TU CÓDIGO AQUÍ
}

In [ ]:
tabla = []
for nombre, m in modelos.items():
    cv = cross_val_score(m, Xc_ent, yc_ent, cv=5)
    prueba = m.fit(Xc_ent, yc_ent).score(Xc_pru, yc_pru)
    tabla.append({"modelo": nombre, "CV media": cv.mean(), "CV desv.": cv.std(), "prueba": prueba})

pd.DataFrame(tabla).sort_values("CV media", ascending=False).round(3)

**Para comentar en grupo:**

- ¿Cuál modelo gana? ¿La diferencia es grande comparada con la desviación de la CV?
- ¿Cuál elegirías si tienes que **explicarle** las decisiones a un médico?
- ¿Cuál elegirías si necesitas predecir **muy rápido** millones de casos?

## 10. 🏆 Reto final: ¿quién aprueba?

Tenemos datos (simulados) de 400 alumnos: horas de estudio por semana, asistencia, semestre que cursan
y si aprobaron. Tu misión:

1. Explora los datos (gráficas, promedios por grupo).
2. Separa entrenamiento y prueba.
3. Prueba **al menos dos** modelos y elige uno con validación cruzada.
4. Reporta la exactitud, la matriz de confusión y el recall de los que **reprueban**
   (son los alumnos a los que queremos ayudar a tiempo).
5. Con Random Forest, ¿qué variable es la más importante?

In [ ]:
# Datos simulados: no son de alumnos reales.
rng = np.random.default_rng(2026)
n = 400
horas = rng.gamma(shape=3, scale=2, size=n).clip(0, 20).round(1)
asistencia = rng.normal(80, 12, size=n).clip(30, 100).round(0)
semestre = rng.integers(1, 9, size=n)
z = 0.45 * (horas - 6) + 0.04 * (asistencia - 80) + 0.05 * (semestre - 4) + rng.normal(0, 1.1, size=n)
aprobo = (z > -0.3).astype(int)

reto = pd.DataFrame({"horas": horas, "asistencia": asistencia, "semestre": semestre, "aprobo": aprobo})
reto.head()

In [ ]:
# TU CÓDIGO AQUÍ
# Agrega todas las celdas que necesites.

---

## Resumen

| Modelo | ¿Escalar? | ¿Fácil de explicar? | Bueno para |
|---|---|---|---|
| Regresión logística | sí | sí | líneas base, probabilidades |
| kNN | **indispensable** | con ejemplos parecidos | datos pequeños |
| Árbol de decisión | no | **mucho** | reglas claras |
| Random Forest | no | importancia de variables | datos en tablas |
| SVM | **indispensable** | poco | muchas variables, texto |
| Naive Bayes | no | más o menos | texto, muy rápido |

**Recuerda:**

- Separa la prueba desde el inicio y úsala **una sola vez**, al final.
- Elige hiperparámetros con **validación cruzada**.
- Si las clases están desbalanceadas, **no confíes solo en la exactitud**.

¡Buen trabajo! 🎉